In [2]:
# =======================
# Imports
# =======================
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 150


# -------------------------
# Where you are (should be .../presentation)
# -------------------------
PRESENTATION_DIR = Path.cwd()
print("CWD:", PRESENTATION_DIR)

GEOMS = ["102string", "160string"]

# Where to save plots + tables
OUTDIR = (PRESENTATION_DIR / "plots_performance_compare").resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)
print("OUTDIR:", OUTDIR)

# File patterns (GraphNeT)
ENERGY_GLOB  = "energy/test_predictions*.csv"
AZIMUTH_GLOB = "azimuth/test_predictions*.csv"
ZENITH_GLOB  = "zenith/test_predictions*.csv"

# Energy binning in log10(E/GeV)
LOGE_BIN_WIDTH = 0.25
MIN_EVENTS_PER_BIN = 50   # skip bins with too few stats

# Detector latitude for sin(δ) proxy (P-ONE site). Update if needed.
DETECTOR_LAT_DEG = 47.77

# Azimuth convention:
# IceTray/I3Direction: azimuth=0 -> South, azimuth=pi/2 -> East
AZIMUTH_ZERO_IS_SOUTH = True


# =======================
# Helpers
# =======================
def _find_one(base: Path, pattern: str) -> Path:
    hits = sorted((base).glob(pattern))
    if len(hits) == 0:
        raise FileNotFoundError(f"No files matched: {base / pattern}")
    return hits[0]

def _choose_merge_key(*dfs: pd.DataFrame) -> str | None:
    candidates = ["event_id", "event_no", "event", "idx", "index"]
    common = set(dfs[0].columns)
    for d in dfs[1:]:
        common &= set(d.columns)
    for k in candidates:
        if k in common:
            return k
    return None

def _first_existing(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns exist: {candidates}")

def _to_radians(arr: np.ndarray) -> np.ndarray:
    m = np.nanmax(arr)
    if m > 2*np.pi + 0.2:
        return np.deg2rad(arr)
    return arr

def _wrap_0_2pi(phi: np.ndarray) -> np.ndarray:
    return np.mod(phi, 2*np.pi)

def compute_sin_delta_from_local(
    th: np.ndarray, ph: np.ndarray, lat_deg: float, azimuth_zero_is_south: bool = True
) -> np.ndarray:
    """sin(δ) from local (zenith, azimuth) + detector latitude."""
    lat = np.deg2rad(lat_deg)

    # altitude a = pi/2 - zenith
    sin_a = np.cos(th)
    cos_a = np.sin(th)

    # Convert azimuth to "from North" if needed (astronomy convention)
    A = ph + np.pi if azimuth_zero_is_south else ph

    sin_delta = np.sin(lat) * sin_a + np.cos(lat) * cos_a * np.cos(A)
    return np.clip(sin_delta, -1.0, 1.0)

def compute_opening_angle_deg(th_true, ph_true, th_pred, ph_pred) -> np.ndarray:
    ux_t = np.sin(th_true) * np.cos(ph_true)
    uy_t = np.sin(th_true) * np.sin(ph_true)
    uz_t = np.cos(th_true)

    ux_p = np.sin(th_pred) * np.cos(ph_pred)
    uy_p = np.sin(th_pred) * np.sin(ph_pred)
    uz_p = np.cos(th_pred)

    dot = ux_t*ux_p + uy_t*uy_p + uz_t*uz_p
    dot = np.clip(dot, -1.0, 1.0)
    return np.degrees(np.arccos(dot))

def build_logE_bins(logE: pd.Series, width: float) -> np.ndarray:
    xmin = np.floor(np.nanmin(logE) / width) * width
    xmax = np.ceil(np.nanmax(logE) / width) * width
    return np.arange(xmin, xmax + width, width)

def binned_quantiles(values: pd.Series, bins: pd.Categorical, qs=(0.16, 0.50, 0.84)) -> pd.DataFrame:
    g = values.groupby(bins, observed=True)
    out = g.quantile(list(qs)).unstack()
    out.columns = [f"q{int(q*100):02d}" for q in qs]
    out["n"] = g.size()
    out["logE_center"] = [iv.mid for iv in out.index]
    return out.reset_index(drop=True)

def _geom_label(g: str) -> str:
    return "102-string" if g.startswith("102") else ("160-string" if g.startswith("160") else g)

def _summ_stats(arr: np.ndarray) -> tuple[float, float, float]:
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (np.nan, np.nan, np.nan)
    return (float(np.quantile(arr, 0.50)), float(np.quantile(arr, 0.16)), float(np.quantile(arr, 0.84)))


# =======================
# Load, merge, compute opening angle, bin
# =======================
per_geom: dict[str, pd.DataFrame] = {}
per_event: dict[str, pd.DataFrame] = {}

for geom in GEOMS:
    base = PRESENTATION_DIR / geom

    energy_path = _find_one(base, ENERGY_GLOB)
    az_path     = _find_one(base, AZIMUTH_GLOB)
    zen_path    = _find_one(base, ZENITH_GLOB)

    e_df   = pd.read_csv(energy_path)
    az_df  = pd.read_csv(az_path)
    zen_df = pd.read_csv(zen_path)

    print(f"\n[{geom}]")
    print("ENERGY path:", energy_path)
    print("AZ path    :", az_path)
    print("ZEN path   :", zen_path)

    key = _choose_merge_key(e_df, az_df, zen_df)
    if key is None:
        raise RuntimeError("Could not find a common merge key among energy/az/zen.")
    print("Merge key:", key)

    # use true_log10_energy if available (it is, in your case)
    if "true_log10_energy" in e_df.columns:
        e_df = e_df[[key, "true_log10_energy"]].copy()
        e_df["log10E_true_GeV"] = e_df["true_log10_energy"].astype(float)
    else:
        # fallback (rare)
        trueE = _first_existing(e_df, ["true_energy", "energy_true", "true_energy_GeV", "true_energy_gev"])
        e_df = e_df[[key, trueE]].copy()
        e_df["log10E_true_GeV"] = np.log10(np.clip(e_df[trueE].astype(float), 1e-12, np.inf))

    # angles (prefer radian columns)
    th_t = _first_existing(zen_df, ["true_zenith_radian", "true_zenith"])
    th_p = _first_existing(zen_df, ["pred_zenith_radian", "pred_zenith"])
    ph_t = _first_existing(az_df,  ["true_azimuth_radian", "true_azimuth"])
    ph_p = _first_existing(az_df,  ["pred_azimuth_radian", "pred_azimuth"])

    # merge
    m = e_df.merge(az_df[[key, ph_t, ph_p]], on=key, how="inner")
    m = m.merge(zen_df[[key, th_t, th_p]], on=key, how="inner")
    print("Merged shape:", m.shape)

    # radians + wrap
    th_true = _to_radians(m[th_t].to_numpy(float))
    th_pred = _to_radians(m[th_p].to_numpy(float))
    ph_true = _wrap_0_2pi(_to_radians(m[ph_t].to_numpy(float)))
    ph_pred = _wrap_0_2pi(_to_radians(m[ph_p].to_numpy(float)))

    # opening angle
    m["opening_deg"] = compute_opening_angle_deg(th_true, ph_true, th_pred, ph_pred)

    # sin(delta) (true)
    m["sin_delta_true"] = compute_sin_delta_from_local(
        th=th_true,
        ph=ph_true,
        lat_deg=DETECTOR_LAT_DEG,
        azimuth_zero_is_south=AZIMUTH_ZERO_IS_SOUTH,
    )

    # store per-event minimal
    per_event[geom] = m[["log10E_true_GeV", "opening_deg", "sin_delta_true"]].copy()

    # bin overall curve
    bins_edges = build_logE_bins(m["log10E_true_GeV"], LOGE_BIN_WIDTH)
    bins = pd.cut(m["log10E_true_GeV"], bins=bins_edges, include_lowest=True)
    qdf = binned_quantiles(m["opening_deg"], bins, qs=(0.16, 0.50, 0.84))
    qdf = qdf[qdf["n"] >= MIN_EVENTS_PER_BIN].copy()
    per_geom[geom] = qdf.sort_values("logE_center").reset_index(drop=True)

    print("Binned rows kept:", len(per_geom[geom]), "| total bins:", len(bins_edges)-1)
    print("opening_deg overall median:", float(np.nanmedian(m["opening_deg"])))


# =======================
# Plot 1: Overall angular resolution vs energy (102 vs 160)
# =======================
fig, ax = plt.subplots(figsize=(6.8, 4.4))

for geom in GEOMS:
    qdf = per_geom[geom]
    x = qdf["logE_center"].to_numpy()
    y = qdf["q50"].to_numpy()
    ylo = qdf["q16"].to_numpy()
    yhi = qdf["q84"].to_numpy()
    ax.plot(x, y, marker="o", ms=3, lw=1.5, label=_geom_label(geom))
    ax.fill_between(x, ylo, yhi, alpha=0.2)

ax.set_xlabel(r"$\log_{10}(E_\mathrm{true}/\mathrm{GeV})$")
ax.set_ylabel(r"Opening angle $\Delta\Psi$ [deg] (median, 16–84%)")
ax.set_title("Angular resolution — 102 vs 160")
ax.grid(True, alpha=0.3)
ax.legend()

out1 = OUTDIR / "angular_resolution_overall_102_vs_160.png"
fig.tight_layout()
fig.savefig(out1)
plt.close(fig)
print("Saved:", out1)


# =======================
# Plot 2: Improvement ratio (median_102 / median_160)
# =======================
g102, g160 = "102string", "160string"
q102 = per_geom[g102]
q160 = per_geom[g160]

# assume same bin centers
x = q102["logE_center"].to_numpy()
ratio = (q102["q50"].to_numpy() / q160["q50"].to_numpy())

fig, ax = plt.subplots(figsize=(6.8, 4.4))
ax.plot(x, ratio, marker="o", ms=3, lw=1.5)
ax.axhline(1.0, lw=1.0, alpha=0.7)
ax.set_xlabel(r"$\log_{10}(E_\mathrm{true}/\mathrm{GeV})$")
ax.set_ylabel(r"$\mathrm{median}_{102} / \mathrm{median}_{160}$")
ax.set_title("Angular-resolution improvement — ratio (102 / 160)")
ax.grid(True, alpha=0.3)

ratio_med = float(np.nanmedian(ratio))
ax.text(
    0.02, 0.95,
    f"median ratio across bins = {ratio_med:.2f}\n(>1 means 160 better)",
    transform=ax.transAxes, va="top",
    bbox=dict(alpha=0.15, edgecolor="none")
)

out2 = OUTDIR / "improvement_ratio_102_over_160.png"
fig.tight_layout()
fig.savefig(out2)
plt.close(fig)
print("Saved:", out2)


# =======================
# Plot 3: Opening-angle histogram in an energy band
# =======================
LOGE_MIN, LOGE_MAX = 4.0, 5.0  # change if you want

def _select_band(df: pd.DataFrame, lo: float, hi: float) -> pd.DataFrame:
    return df[(df["log10E_true_GeV"] >= lo) & (df["log10E_true_GeV"] < hi)].copy()

e102 = _select_band(per_event[g102], LOGE_MIN, LOGE_MAX)
e160 = _select_band(per_event[g160], LOGE_MIN, LOGE_MAX)

fig, ax = plt.subplots(figsize=(6.8, 4.4))
bins = np.linspace(0, 30, 61)  # 0..30 deg, 0.5 deg bins

ax.hist(e102["opening_deg"], bins=bins, density=True, alpha=0.35, label=f"102 (n={len(e102)})")
ax.hist(e160["opening_deg"], bins=bins, density=True, alpha=0.35, label=f"160 (n={len(e160)})")

ax.set_xlabel(r"Opening angle $\Delta\Psi$ [deg]")
ax.set_ylabel("Normalized density")
ax.set_title(f"Opening-angle distribution — {LOGE_MIN:.1f} ≤ log10(E/GeV) < {LOGE_MAX:.1f}")
ax.grid(True, alpha=0.3)
ax.legend()

m50_102, m16_102, m84_102 = _summ_stats(e102["opening_deg"].to_numpy())
m50_160, m16_160, m84_160 = _summ_stats(e160["opening_deg"].to_numpy())

ax.text(
    0.98, 0.95,
    "Medians (16–84%)\n"
    f"102: {m50_102:.2f} ({m16_102:.2f}–{m84_102:.2f})\n"
    f"160: {m50_160:.2f} ({m16_160:.2f}–{m84_160:.2f})",
    transform=ax.transAxes, va="top", ha="right",
    bbox=dict(alpha=0.15, edgecolor="none")
)

out3 = OUTDIR / f"opening_angle_hist_logE_{LOGE_MIN:.1f}_{LOGE_MAX:.1f}_102_vs_160.png"
fig.tight_layout()
fig.savefig(out3)
plt.close(fig)
print("Saved:", out3)


# =======================
# Plot 4: sin(delta) bands (4 separate PNG)
# =======================
SINDELTA_BANDS = [(-1.0, -0.5), (-0.5, 0.0), (0.0, 0.5), (0.5, 1.0)]

def _binned_curve_from_events(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame(columns=["q16", "q50", "q84", "n", "logE_center"])
    bins_edges = build_logE_bins(df["log10E_true_GeV"], LOGE_BIN_WIDTH)
    bins = pd.cut(df["log10E_true_GeV"], bins=bins_edges, include_lowest=True)
    qdf = binned_quantiles(df["opening_deg"], bins, qs=(0.16, 0.50, 0.84))
    qdf = qdf[qdf["n"] >= MIN_EVENTS_PER_BIN].copy()
    return qdf.sort_values("logE_center").reset_index(drop=True)

for i, (lo, hi) in enumerate(SINDELTA_BANDS, start=1):
    d102 = per_event[g102]
    d160 = per_event[g160]

    d102 = d102[(d102["sin_delta_true"] >= lo) & (d102["sin_delta_true"] < hi)]
    d160 = d160[(d160["sin_delta_true"] >= lo) & (d160["sin_delta_true"] < hi)]

    qd102 = _binned_curve_from_events(d102)
    qd160 = _binned_curve_from_events(d160)

    fig, ax = plt.subplots(figsize=(6.8, 4.4))

    for geom, qd in [(g102, qd102), (g160, qd160)]:
        if len(qd) == 0:
            continue
        ax.plot(qd["logE_center"], qd["q50"], marker="o", ms=3, lw=1.5, label=_geom_label(geom))
        ax.fill_between(qd["logE_center"], qd["q16"], qd["q84"], alpha=0.2)

    ax.set_xlabel(r"$\log_{10}(E_\mathrm{true}/\mathrm{GeV})$")
    ax.set_ylabel(r"Opening angle $\Delta\Psi$ [deg] (median, 16–84%)")
    ax.set_title(f"Angular resolution — sin(δ) in [{lo:.1f}, {hi:.1f})")
    ax.grid(True, alpha=0.3)
    ax.legend()

    out = OUTDIR / f"angular_resolution_sindelta_bin{i}_102_vs_160.png"
    fig.tight_layout()
    fig.savefig(out)
    plt.close(fig)
    print("Saved:", out)


# =======================
# Numeric summary for slide captions
# =======================
# Overall medians (all energies)
all102 = per_event[g102]["opening_deg"].to_numpy()
all160 = per_event[g160]["opening_deg"].to_numpy()
med102, p16_102, p84_102 = _summ_stats(all102)
med160, p16_160, p84_160 = _summ_stats(all160)

# Improvement in percent using medians
impr_pct = (1.0 - (med160 / med102)) * 100.0 if np.isfinite(med102) and med102 > 0 else np.nan

# Energy-band medians (same band as histogram)
band102 = e102["opening_deg"].to_numpy()
band160 = e160["opening_deg"].to_numpy()
bmed102, b16_102, b84_102 = _summ_stats(band102)
bmed160, b16_160, b84_160 = _summ_stats(band160)
band_impr_pct = (1.0 - (bmed160 / bmed102)) * 100.0 if np.isfinite(bmed102) and bmed102 > 0 else np.nan

summary_path = OUTDIR / "summary_102_vs_160.txt"
with open(summary_path, "w") as f:
    f.write("102 vs 160 — Angular resolution summary\n")
    f.write("=====================================\n\n")
    f.write("Overall (all energies):\n")
    f.write(f"  102 median ΔΨ = {med102:.3f} deg (16–84%: {p16_102:.3f}–{p84_102:.3f})\n")
    f.write(f"  160 median ΔΨ = {med160:.3f} deg (16–84%: {p16_160:.3f}–{p84_160:.3f})\n")
    f.write(f"  Relative improvement (median) = {impr_pct:.1f}% (160 better if positive)\n\n")
    f.write(f"Energy band {LOGE_MIN:.1f} ≤ log10(E/GeV) < {LOGE_MAX:.1f}:\n")
    f.write(f"  102 median ΔΨ = {bmed102:.3f} deg (16–84%: {b16_102:.3f}–{b84_102:.3f})\n")
    f.write(f"  160 median ΔΨ = {bmed160:.3f} deg (16–84%: {b16_160:.3f}–{b84_160:.3f})\n")
    f.write(f"  Relative improvement (median) = {band_impr_pct:.1f}% (160 better if positive)\n\n")
    f.write("Ratio curve:\n")
    f.write(f"  median over bins of (median_102 / median_160) = {ratio_med:.3f}\n")

print("Saved:", summary_path)

CWD: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation
OUTDIR: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/plots_performance_compare

[102string]
ENERGY path: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/102string/energy/test_predictions.csv
AZ path    : /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/102string/azimuth/test_predictions.csv
ZEN path   : /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/102string/zenith/test_predictions.csv
Merge key: event_id


Merged shape: (20272, 7)
Binned rows kept: 20 | total bins: 20
opening_deg overall median: 5.36786200117464

[160string]
ENERGY path: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/160string/energy/test_predictions.csv
AZ path    : /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/160string/azimuth/test_predictions.csv
ZEN path   : /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/160string/zenith/test_predictions.csv
Merge key: event_id
Merged shape: (23889, 7)
Binned rows kept: 20 | total bins: 20
opening_deg overall median: 4.434443697327626
Saved: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/plots_performance_compare/angular_resolution_overall_102_vs_160.png
Saved: /project/6061446/kbas/Graphnet-Applications/Training/EventPulseSeries_nonoise/presentation/plots_performance_compare/improvement_ratio_102_over_160.pn